# Microsoft Agent Framework — Analyst → Strategist workflow

Microsoft merged **both** AutoGen and Semantic Kernel into a single SDK,
**Microsoft Agent Framework (MAF)**, which reached GA in April 2026. It keeps AutoGen's
multi-agent conversation patterns and Semantic Kernel's enterprise plumbing (plugins,
middleware, observability), with native support for Anthropic, OpenAI, Azure, Bedrock,
and others.

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| LangGraph | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| CrewAI | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| **Microsoft Agent Framework** | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |



In [9]:
%pip install -q agent-framework-core==1.18.0 agent-framework-anthropic==1.0.0b260910 agent-framework-orchestrations==1.1.1

After the install cell finishes for the first time, use
> **Runtime ▸ Restart session** before running the rest of the notebook. Colab can keep
> an old, incompatible copy of a dependency loaded in memory even after a fresh install
> replaces it on disk.

> MAF's provider packages are still on beta version numbers and move fast — the pins above are the versions this notebook was tested against, request *and* response paths. An earlier draft of this notebook pinned an older `agent-framework-anthropic` (from the `agent-framework` metapackage's own default) that constructs its response object with a keyword argument current `agent-framework-core` no longer accepts — it looks fine until a real API call succeeds, then fails on `ChatResponse.__init__() got an unexpected keyword argument 'model_id'`. If you see that error, it means your installed `agent-framework-anthropic` predates this fix — rerun the install cell to get the pinned version above. An unpinned `pip install agent-framework` also works but pulls in ~20 extra provider packages (Azure, Bedrock, Gemini, etc.) you don't need for this code.


In [10]:
import os

def _load(key, default=None):
  """Colab Secrets first, then environment variable, then interactive prompt."""
  try:
    from google.colab import userdata
    val = userdata.get(key)
    if val:
      return val
  except Exception:
    pass
  val = os.environ.get(key, default)
  if not val:
    from getpass import getpass
    val = getpass(f"{key}: ")
  return val

ANTHROPIC_API_KEY = _load("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL = _load("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print(f"Using model: {ANTHROPIC_MODEL}")


Using model: claude-sonnet-4-5-20250929


## The scenario (same one from tonight's opener)

Recall the Slide 3 scenario: an assistant reads user interviews, support tickets, and
competitor notes, and is supposed to turn that into a feature recommendation — without
mismatched analysis or hallucinated suggestions.

We'll build that as a **two-step pipeline**:

1. **Analyst** — reads the raw signals, extracts 2–3 themes, and cites which signal(s)
   support each one (a small guardrail against fabrication).
2. **Strategist** — takes *only* the Analyst's themes and recommends ONE feature with a
   one-line rationale. It's told not to invent anything the Analyst didn't surface.

Same signals, same two-step shape, three different frameworks — that's the point of
building all three tonight.

In MAF terms: two **agents** built from one `AnthropicClient`, wired with a pre-built workflow
pattern, so there's no manual graph to draw. We wire them two ways:

| | **Version A: `SequentialBuilder`** | **Version B: `HandoffBuilder`** |
|---|---|---|
| Flow | Fixed: Analyst → Strategist, once | Agents decide at runtime who speaks next |
| Can the Strategist ask for more? | No | Yes: it can hand the conversation **back** to the Analyst |
| How control moves | The builder passes output along | Each agent gets auto-generated `handoff_to_<Agent>` tools and calls one |
| When it ends | After the last participant | When a `termination_condition` you write returns `True` |


In [11]:
SIGNALS = """
- User interview (Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Support ticket #4821: "Can't find where to invite my team members. Confusing menu."
- Competitor teardown (RivalApp): just shipped a guided setup checklist with a progress bar.
- Support ticket #4903: "Wish there was a quick tour when I first logged in."
""".strip()

print(SIGNALS)


- User interview (Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Support ticket #4821: "Can't find where to invite my team members. Confusing menu."
- Competitor teardown (RivalApp): just shipped a guided setup checklist with a progress bar.
- Support ticket #4903: "Wish there was a quick tour when I first logged in."


In [12]:
from agent_framework.anthropic import AnthropicClient

client = AnthropicClient(model=ANTHROPIC_MODEL)

analyst = client.as_agent(
  name="Analyst",
  instructions=(
    "You are a product signal analyst. Extract 2-3 themes from the signals "
    "you're given as bullet points. Cite which signal supports each theme."
  ),
)

strategist = client.as_agent(
  name="Strategist",
  instructions=(
    "You are a product strategist. Using ONLY the themes you're given, "
    "recommend ONE feature to prioritize next sprint with a one-line rationale. "
    "Do not invent new signals."
  ),
)


## Version A: `SequentialBuilder`

A fixed pipeline: the Analyst speaks once, then the Strategist speaks once.

In [13]:
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(participants=[analyst, strategist]).build()

result = await workflow.run(f"Signals:\n{SIGNALS}")
for output in result.get_outputs():
  print(getattr(output, "text", output))
  print("---")


# Recommended Feature for Next Sprint

**Guided onboarding checklist with progress tracking**

**Rationale:** Directly addresses setup abandonment (Maria's pain point), provides the guidance requested in multiple support tickets, and prevents us from falling behind competitors who are already shipping this standard feature.
---


## Version B: `HandoffBuilder`, where the Strategist can send work back

**What changes:**

- **Handoff routes:** `.add_handoff(analyst, [strategist])` and `.add_handoff(strategist, [analyst])`.
  MAF turns each route into a tool (`handoff_to_Strategist`, `handoff_to_Analyst`) that it gives
  to the agent. Calling the tool passes the whole conversation to the other agent. The agent's
  `description` becomes the tool description, which is how each agent knows when a handoff fits.
- **Start agent:** `.with_start_agent(analyst)`.
- **Autonomous mode:** by default, an agent that answers *without* handing off pauses the workflow
  to wait for a human reply. `.with_autonomous_mode()` makes it continue on its own instead.
  `turn_limits` caps how long each agent can keep going.
- **Termination condition:** with autonomous mode on, nothing stops the loop by default.
  We end the run once the Strategist posts a line beginning `FINAL RECOMMENDATION:`, with a
  hard cap on total turns as a safety net.

**Making the hand-back happen:** with these four clear signals, a capable model would usually be
satisfied on the first pass, and you'd never see the loop. So the Analyst is told to do a **quick
first pass** (only the single strongest theme, to save tokens), and the Strategist is told it needs
**at least two independent themes** before committing. That's a realistic pattern: a cheap triage
first, then more depth only when the decision-maker asks for it.

> **Requirement:** handoff participants must be built with
> `require_per_service_call_history_persistence=True`. `build()` raises a `ValueError` without it,
> so these agents are created with `Agent(...)` directly. The helper below falls back to
> creating the agent without the flag on older versions that don't support it.

In [14]:
from agent_framework import Agent

def make_handoff_agent(**kwargs):
  try:
    return Agent(client=client, require_per_service_call_history_persistence=True, **kwargs)
  except TypeError:  # older agent-framework-core without this flag
    return Agent(client=client, **kwargs)

FINAL_MARKER = "FINAL RECOMMENDATION:"

analyst_h = make_handoff_agent(
  name="Analyst",
  description="Product signal analyst. Hand off here when more or deeper themes are needed from the raw signals.",
  instructions=(
    "You are a product signal analyst. Extract themes from the signals as bullet points, "
    "citing which signal supports each theme.\n"
    "- FIRST PASS: report only the ONE strongest theme (quick triage), then hand off to the Strategist.\n"
    "- If the Strategist hands back asking for more, add 1-2 NEW themes not already listed, "
    "each with citations, then hand off to the Strategist again.\n"
    "Never recommend features yourself."
  ),
)

strategist_h = make_handoff_agent(
  name="Strategist",
  description="Product strategist. Hand off here when themes are ready to be turned into a recommendation.",
  instructions=(
    "You are a product strategist. Use ONLY the themes the Analyst has provided; do not invent signals.\n"
    "- You need AT LEAST TWO independent themes, each with a citation, before recommending anything.\n"
    "- If you have fewer, briefly say what is missing and hand off to the Analyst to request more themes. "
    "Do this at most once.\n"
    f"- Once you have enough, write ONE feature recommendation on a line starting with '{FINAL_MARKER}' "
    "followed by a one-line rationale. Do not hand off after that."
  ),
)

In [15]:
from agent_framework.orchestrations import HandoffBuilder

MAX_TURNS = 8  # safety net: total agent replies before forcing a stop

def _is_assistant(msg) -> bool:
  role = getattr(msg, "role", "")
  return getattr(role, "value", role) == "assistant"

def done(conversation) -> bool:
  """Stop when the Strategist posts its final recommendation, or after MAX_TURNS replies."""
  replies = [m for m in conversation if _is_assistant(m)]
  finished = any(
    m.author_name == "Strategist" and FINAL_MARKER in (m.text or "") for m in replies
  )
  return finished or len(replies) >= MAX_TURNS

handoff_workflow = (
  HandoffBuilder(
    name="analyst_strategist_handoff",
    participants=[analyst_h, strategist_h],
    termination_condition=done,
  )
  .with_start_agent(analyst_h)
  .add_handoff(analyst_h, [strategist_h])
  .add_handoff(strategist_h, [analyst_h])  # the hand-back route
  .with_autonomous_mode(turn_limits={"Analyst": 3, "Strategist": 3})
  .build()
)

In [16]:
handoff_trail = []        # (source, target) for every handoff
transcript = {}           # response_id -> [author, text]

async for event in handoff_workflow.run(f"Signals:\n{SIGNALS}", stream=True):
  if event.type == "handoff_sent":
    handoff_trail.append((event.data.source, event.data.target))
    print(f"\n>>> HANDOFF: {event.data.source} -> {event.data.target}\n")
  elif event.type == "output":
    update = event.data
    text = getattr(update, "text", None)
    if not text:
      continue  # tool calls (including handoffs) carry no text
    rid = getattr(update, "response_id", None) or id(update)
    if rid not in transcript:
      transcript[rid] = [getattr(update, "author_name", None) or "agent", ""]
      print(f"\n[{transcript[rid][0]}] ", end="")
    transcript[rid][1] += text
    print(text, end="", flush=True)

print("\n\n" + "=" * 70)
print("Handoff trail:", " -> ".join([handoff_trail[0][0]] + [t for _, t in handoff_trail]) if handoff_trail else "(none)")
went_back = ("Strategist", "Analyst") in handoff_trail
print("Strategist handed back to Analyst:", "YES" if went_back else "no (themes were sufficient first time)")


>>> HANDOFF: Analyst -> Strategist


[Strategist] I need
[Strategist]  to gather
[Strategist]  themes
[Strategist]  from these signals before
[Strategist]  I can make
[Strategist]  a
[Strategist]  recommendation. Let me hand
[Strategist]  off to the Analyst to
[Strategist]  identify
[Strategist]  the underlying
[Strategist]  themes.
>>> HANDOFF: Strategist -> Analyst


[Analyst] **
[Analyst] Strongest
[Analyst]  Theme (First
[Analyst]  Pass):**
[Analyst] 

• **
[Analyst] On
[Analyst] boarding friction
[Analyst]  is
[Analyst]  causing
[Analyst]  user drop
[Analyst] -off** - Users
[Analyst]  are
[Analyst]  struggling
[Analyst]  to complete initial
[Analyst]  setup and
[Analyst]  understand
[Analyst]  how
[Analyst]  to get
[Analyst]  started with
[Analyst]  the product.
[Analyst] 
  
[Analyst] - User
[Analyst]  interview (
[Analyst] Maria, SMB owner): "I
[Analyst]  got lost during setup and gave up before
[Analyst]  finishing."
  - Support ticket #
[Analyst] 4821: "Can't find where
[Anal

MAF is a young framework especially when compared to years of production mileage for LangGraph and CrewAI. It is promising and worth watching, but **not yet the safe default for a production system** the way the other two are.

**Sequential vs. Handoff:** `SequentialBuilder` is right when the steps are fixed. It's cheap
and predictable. `HandoffBuilder` suits a conversation whose path depends on what the agents find,
such as customer-support triage or a reviewer sending work back. In return you need a
termination condition and turn limits, because the agents decide when to move on.

In LangGraph, the loop back was an explicit
conditional edge *we* wrote. In CrewAI's hierarchical mode, a manager decided. Here, each agent
decides for itself by calling a handoff tool.
